# Links

GitHub repository:
https://github.com/nylandgit/IND320-Repo

Streamlit app:
https://nylandgit-ind320-repo.streamlit.app/

Code runthrough video:


# References

In the interest of avoiding claims of self-plagiarism (don't know if this is necessary for IND320?):

Bulks of the following code is copied and adapted from Thor Martin Nyland's Project Work Part 1. Parts of the code is copied directly without any modification. The AI Statement is copied directly, as there has been no change in AI usage since Part 1.

Parts of the code have been copied from khliland's IND320 GitHub Repository. These parts are highlighted where applicable.

# AI Statement and Descriptive Log

## AI Statement

This project has used the free VS Code Copilot generative AI to: Generate code, identify bugs, fix bugs, brainstorm technical solutions, and to teach the process of coding through in-depth explanations of each generated line of code. Some Google searches have also involved Google's built-in generative AI.

The project author has steered the AI's coding through accepting, rejecting, and modifying the AI's code based on the author's intuition. The author has requested code explanations, studied the AI's code, and made explanatory comments for each code block based on the author's understanding.

The code provided is therefore created partly by the author and partly by generative AI. The explanatory code comments are written exclusively by the author.

## Descriptive Log

### Section 1: Non-Streamlit Code
* Copied and adapted the Atlas NVE shapefiles sampling example from class. Developed working demo for Precipitation Area sampling.
* Developed working demo for Electricity Area sampling.
* Developed working demo(s) for Open-Meteo API data retreival based on Electricity Area samples. Built function download_region(region, start_date, end_date). The code is bloated and uncommented, will revise and explain code after plots are made and I understand the project requirements more deeply.

<p></p>
<p></p>

### Text
* Text


# Non-Streamlit Code

## 1. Atlas NVE

### 1.1 WIP Precipitation Area Sampling

In [1]:
# This code has been copied and adapted from Kristian Hovde Lilands IND320
# lecture held at NMBU on 26.10.2026.
import geopandas as gpd

# Read file
gdf_precip = gpd.read_file("../shapefiles/precipitation areas/NVE_Region.shp")

# Generate random points
n = 1000
points_precip = gdf_precip['geometry'].sample_points(size=n, random_state=42)

# Transform WGS84 (EPSG:4326) for latitude and longitude (?)
points_wgs84_precip = points_precip.to_crs(epsg=4326)

# Extract coordinates and print
for idx_precip, multipoint_precip in points_wgs84_precip.items():
    region_name_precip = gdf_precip.loc[idx_precip, 'REGION']
    #print(f"Region: {region_name_precip}")

    for point_precip in multipoint_precip.geoms:
        # Do something more useful with the coordinates than this (?)
        lat_precip = point_precip.y
        lon_precip = point_precip.x

print(lat_precip)


60.25201081365595


### 1.2 WIP Electricity Areas Sampling

In [2]:
# This code has been copied and adapted from Kristian Hovde Lilands IND320
# lecture held at NMBU on 26.10.2026.

# Read file
gdf_el = gpd.read_file("../shapefiles/electricity areas/Budområder.shp")

# Generate random points
n = 1000
points_el = gdf_el['geometry'].sample_points(size=n, random_state=42)

# Transform WGS84 (EPSG:4326) for latitude and longitude (?)
points_wgs84_el = points_el.to_crs(epsg=4326)

# Extract coordinates and save them
locations_by_region_el = {}
for idx_el, multipoint_el in points_wgs84_el.items():
    region_name_el = gdf_el.loc[idx_el, 'budomr']
    print(f"Region: {region_name_el}")

    for point_el in multipoint_el.geoms:
        # Do something more useful with the coordinates than this (?)
        lat_el = point_el.y
        lon_el = point_el.x
        locations_by_region_el.setdefault(region_name_el, []).append((lat_el, lon_el))

print(lat_el)
print(gdf_el.head())


Region: NO 1
Region: NO 2
Region: NO 3
Region: NO 4
Region: NO 5
61.420682293207406
   objectid budomr  globalid    st_area(sh    st_perimet  \
0       1.0   NO 1       NaN  6.052902e+10  1.657761e+06   
1       2.0   NO 2       NaN  6.875969e+10  1.256473e+06   
2       3.0   NO 3       NaN  1.042199e+11  1.773323e+06   
3       4.0   NO 4       NaN  2.032500e+11  3.385499e+06   
4       5.0   NO 5       NaN  3.334870e+10  1.106762e+06   

                                            geometry  
0  POLYGON ((337473.63 6970542.97, 337711.9 69703...  
1  POLYGON ((-9833.91 6726499.86, -9836 6726404.0...  
2  POLYGON ((308588.7 7261519.43, 313576.63 72590...  
3  POLYGON ((973920.93 7962739.19, 974922.77 7962...  
4  POLYGON ((101466.43 6881318.54, 101525.43 6881...  


### 1.3 Open-Meteo API: NO 3, 2024

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests


def download_region(region, start_date, end_date):
    locations = locations_by_region_el[region]
    url = "https://archive-api.open-meteo.com/v1/archive"
    cache_dir = Path("open_meteo_cache")
    cache_dir.mkdir(exist_ok=True)
    results = []
    end_date = pd.Timestamp(end_date)
    periods = pd.date_range(start_date, end_date, freq="14D")
    batches_per_period = (len(locations) + 49) // 50
    total_batches = len(periods) * batches_per_period
    completed_batches = 0
    print(f"Starting {region}: {len(locations)} locations, {total_batches} batches.", flush=True)

    for period_number, period_start in enumerate(periods, start=1):
        period_end = min(period_start + pd.Timedelta(days=13), end_date)
        for batch_start in range(0, len(locations), 50):
            batch = locations[batch_start:batch_start + 50]
            params = {
                "latitude": ",".join(str(lat) for lat, lon in batch),
                "longitude": ",".join(str(lon) for lat, lon in batch),
                "start_date": period_start.strftime("%Y-%m-%d"),
                "end_date": period_end.strftime("%Y-%m-%d"),
                "hourly": "temperature_2m,precipitation",
                "models": "era5",
                "timezone": "UTC",
            }
            cache_file = cache_dir / (
                f"{region.replace(' ', '')}_{batch_start}_"
                f"{period_start:%Y%m%d}_{period_end:%Y%m%d}.json"
            )
            cached = cache_file.exists()
            batch_number = completed_batches + 1
            print(
                f"[{batch_number}/{total_batches}] {period_start:%Y-%m-%d} to "
                f"{period_end:%Y-%m-%d}, locations "
                f"{batch_start + 1}-{batch_start + len(batch)}: "
                f"{'loading cache' if cached else 'requesting API'}.",
                flush=True,
            )

            if cached:
                print(f"[{batch_number}/{total_batches}] Reading {cache_file}.", flush=True)
                with cache_file.open(encoding="utf-8") as file:
                    api_results = json.load(file)
                print(f"[{batch_number}/{total_batches}] Cache loaded.", flush=True)
            else:
                for attempt in range(10):
                    print(f"[{batch_number}/{total_batches}] API attempt {attempt + 1}/10.", flush=True)
                    response = requests.get(url, params=params, timeout=120)
                    if response.status_code == 429:
                        delay = min(2 ** (attempt + 1), 60)
                        print(f"Rate limited; retrying in {delay} seconds.", flush=True)
                        time.sleep(delay)
                        continue
                    response.raise_for_status()
                    api_results = response.json()
                    print(f"[{batch_number}/{total_batches}] API response received.", flush=True)
                    time.sleep(2)
                    break
                else:
                    response.raise_for_status()

            if isinstance(api_results, dict):
                api_results = [api_results]
            if len(api_results) != len(batch):
                raise ValueError("The API returned an unexpected number of locations.")

            print(f"[{batch_number}/{total_batches}] Validating {len(api_results)} locations.", flush=True)
            expected_hours = (period_end - period_start).days * 24 + 24
            for (lat, lon), location_data in zip(batch, api_results):
                hourly = pd.DataFrame(location_data["hourly"])
                if len(hourly) != expected_hours:
                    raise ValueError(f"Incomplete hourly data for {lat}, {lon}.")
                hourly["region"] = region
                hourly["latitude"] = lat
                hourly["longitude"] = lon
                results.append(hourly)

            if not cached:
                with cache_file.open("w", encoding="utf-8") as file:
                    json.dump(api_results, file)
                print(f"[{batch_number}/{total_batches}] Saved cache.", flush=True)

            completed_batches += 1
            print(f"[{completed_batches}/{total_batches}] Batch complete.", flush=True)

        print(f"Completed period {period_number}/{len(periods)}.", flush=True)

    print(f"Finished {region}: {completed_batches} batches complete.", flush=True)
    return pd.concat(results, ignore_index=True)


no3_2024 = download_region("NO 3", "2024-01-01", "2024-12-31")


### 1.4 WIP Plots